# **Recap of [Lesson 4](https://github.com/dlt-hub/dlt/blob/master/docs/education/dlt-fundamentals-course/lesson_4_using_pre_build_sources_and_destinations.ipynb) 👩‍💻🚀**

1. Listed all available verified sources.
2. Initialized the `github_api` verified source.
3. Explored the built-in `rest_api` source.
4. Explored the built-in `sql_database` source.
5. Explored the built-in `filesystem` source.
6. Learned how to switch between destinations.


---

# **Write Disposition and Incremental Loading** ⚙️🧠 [![Open in molab](https://marimo.io/molab-shield.svg)](https://molab.marimo.io/github/dlt-hub/dlt/blob/master/docs/education/dlt-fundamentals-course/lesson_5_write_disposition_and_incremental_loading.py) [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/dlt-hub/dlt/blob/master/docs/education/dlt-fundamentals-course/lesson_5_write_disposition_and_incremental_loading.ipynb) [![GitHub badge](https://img.shields.io/badge/github-view_source-2b3137?logo=github)](https://github.com/dlt-hub/dlt/blob/master/docs/education/dlt-fundamentals-course/lesson_5_write_disposition_and_incremental_loading.ipynb)


**Here, you will learn:**
- `dlt` write dispositions:
  - Append
  - Replace
  - Merge
- What incremental loading is
- How to update and deduplicate your data

---
## **`dlt` write dispositions**

A **write disposition** in the context of the `dlt` library defines how data should be written to the destination. There are three types:

- **Append**: The **default** disposition. It appends new data to the existing data in the destination.

- **Replace**: This disposition replaces all existing data at the destination with the new data from the resource. It **deletes** all previous data and **recreates** the schema before loading.

- **Merge**: This disposition merges incoming data with existing data at the destination. For `merge`, you must specify a `primary_key` for the resource.

The choice of write disposition depends on your dataset and how you extract it. For more details, refer to the [Incremental loading page](https://dlthub.com/docs/general-usage/incremental-loading).

You can specify a `write_disposition` in the resource decorator:

```python
@dlt.resource(write_disposition="append")
def my_resource():
  ...
  yield data
```

Or directly in the pipeline run:

```python
load_info = pipeline.run(my_resource, write_disposition="replace")
```

> If both are specified, the write disposition at the pipeline run level overrides the one set at the resource level.

In [1]:
%%capture
!pip install "dlt[duckdb]"

---
### **1. Append**

As we have already said, `append` is the default loading behavior. Now we will explore how this write disposition works.

Let's remember our Quick Start data sample with pokemons:

In [2]:
# Sample data containing pokemon details
data = [
    {"id": "1", "name": "bulbasaur", "size": {"weight": 6.9, "height": 0.7}},
    {"id": "4", "name": "charmander", "size": {"weight": 8.5, "height": 0.6}},
    {"id": "25", "name": "pikachu", "size": {"weight": 6, "height": 0.4}},
]

We create a `dlt` pipeline as usual and load this data into DuckDB.

In [4]:
import dlt
from dlt.common.typing import TDataItems


@dlt.resource(
    name="pokemon",
    write_disposition="append",
)
def append_pokemon() -> TDataItems:
    yield data


append_pipeline = dlt.pipeline(
    pipeline_name="append_poke_pipeline",
    destination="duckdb",
    dataset_name="pokemon_data",
)

load_info = append_pipeline.run(append_pokemon)
print(load_info)

# explore loaded data
append_pipeline.dataset().pokemon.df()

Pipeline append_poke_pipeline load step finished in 0.14 seconds
1 load package(s) were loaded to destination duckdb and into dataset pokemon_data
The duckdb destination used duckdb:////content/append_poke_pipeline.duckdb location to store data
Load package 1795193224.2347627 is LOADED and contains no failed jobs


,id,name,size__weight,size__height,_dlt_load_id,_dlt_id
0,1,bulbasaur,6.9,0.7,1795193119.437103,VQ1nfSF5COa7Jw
1,4,charmander,8.5,0.6,1795193119.437103,aA00nesdVZHMeg
2,25,pikachu,6.0,0.4,1795193119.437103,PGKxaGMbiUaDhw
3,1,bulbasaur,6.9,0.7,1795193224.2347627,kwHrPNei1w2y+Q
4,4,charmander,8.5,0.6,1795193224.2347627,rOp8XsNn9sTwGA
5,25,pikachu,6.0,0.4,1795193224.2347627,KU2333J0fR3Kzw


Run this example **twice**, and you'll notice that each time a copy of the data is added to your tables. We call this load mode **append**, and it is very useful.

Example use case: when you have a new folder created daily with JSON log files, and you want to ingest them incrementally.


In [5]:
load_info = append_pipeline.run(append_pokemon)
print(load_info)

# explore loaded data
append_pipeline.dataset().pokemon.df()

Pipeline append_poke_pipeline load step finished in 0.19 seconds
1 load package(s) were loaded to destination duckdb and into dataset pokemon_data
The duckdb destination used duckdb:////content/append_poke_pipeline.duckdb location to store data
Load package 1795193233.1695683 is LOADED and contains no failed jobs


,id,name,size__weight,size__height,_dlt_load_id,_dlt_id
0,1,bulbasaur,6.9,0.7,1795193119.437103,VQ1nfSF5COa7Jw
1,4,charmander,8.5,0.6,1795193119.437103,aA00nesdVZHMeg
2,25,pikachu,6.0,0.4,1795193119.437103,PGKxaGMbiUaDhw
3,1,bulbasaur,6.9,0.7,1795193224.2347627,kwHrPNei1w2y+Q
4,4,charmander,8.5,0.6,1795193224.2347627,rOp8XsNn9sTwGA
5,25,pikachu,6.0,0.4,1795193224.2347627,KU2333J0fR3Kzw
6,1,bulbasaur,6.9,0.7,1795193233.1695683,8114+H6Jy749fA
7,4,charmander,8.5,0.6,1795193233.1695683,6P22smNEl6fFuQ
8,25,pikachu,6.0,0.4,1795193233.1695683,2d0mFWqmaI3EHg


---
### **2. Replace**

Perhaps this duplicated data is not what you want in your work projects. For example, if your data was updated, how can we refresh it in the database? One way is to tell `dlt` to **replace** the data in the existing tables by using a **write_disposition**.


In [6]:
import dlt


@dlt.resource(
    name="pokemon",
    write_disposition="replace",
)
def replace_pokemon() -> TDataItems:
    yield data


replace_pipeline = dlt.pipeline(
    pipeline_name="replace_poke_pipeline",
    destination="duckdb",
    dataset_name="pokemon_data",
)

load_info = replace_pipeline.run(replace_pokemon)
print(load_info)

# explore loaded data
replace_pipeline.dataset().pokemon.df()

Pipeline replace_poke_pipeline load step finished in 0.30 seconds
1 load package(s) were loaded to destination duckdb and into dataset pokemon_data
The duckdb destination used duckdb:////content/replace_poke_pipeline.duckdb location to store data
Load package 1795193245.299125 is LOADED and contains no failed jobs


,id,name,size__weight,size__height,_dlt_load_id,_dlt_id
0,1,bulbasaur,6.9,0.7,1795193245.299125,vqO+YdC2Z5QZkw
1,4,charmander,8.5,0.6,1795193245.299125,FRIRQFp2fO0HVA
2,25,pikachu,6.0,0.4,1795193245.299125,B5OSee50XF47MA


Run it again:

In [7]:
load_info = replace_pipeline.run(replace_pokemon)
print(load_info)

# explore loaded data
replace_pipeline.dataset().pokemon.df()

Pipeline replace_poke_pipeline load step finished in 0.16 seconds
1 load package(s) were loaded to destination duckdb and into dataset pokemon_data
The duckdb destination used duckdb:////content/replace_poke_pipeline.duckdb location to store data
Load package 1795193250.3467853 is LOADED and contains no failed jobs


,id,name,size__weight,size__height,_dlt_load_id,_dlt_id
0,1,bulbasaur,6.9,0.7,1795193250.3467853,F2FlnX9aSkA1kw
1,4,charmander,8.5,0.6,1795193250.3467853,yvdrrrFeDDbyHQ
2,25,pikachu,6.0,0.4,1795193250.3467853,dr7/BrEgTD25kw


TADA! No duplicates, your data was [fully refreshed](https://dlthub.com/docs/general-usage/full-loading).

---
### **3. [Merge](https://dlthub.com/docs/general-usage/incremental-loading#merge-incremental-loading)**

Consider a scenario where the data in the source has been updated, but you want to avoid reloading the entire dataset.



Merge write disposition is used to merge new data into the destination, using a `merge_key` and/or **deduplicating**/**upserting** new data using a `primary_key`.



![Lesson_5_Write_disposition_and_incremental_loading_img1](https://storage.googleapis.com/dlt-blog-images/dlt-fundamentals-course/Lesson_5_Write_disposition_and_incremental_loading_img1.jpeg)


The **merge** write disposition can be useful in several situations:

1.  If you have a dataset where records are frequently updated and you want to reflect these changes in your database, the `merge` write disposition can be used. It will **update the existing records** with the new data instead of creating duplicate entries.

2. If your data source occasionally sends **duplicate records**, the merge write disposition can help handle this. It uses a `primary_key` to identify unique records, so if a duplicate record (with the same `primary_key`) is encountered, it will be merged with the existing record instead of creating a new one.

3. If you are dealing with **Slowly Changing Dimensions** (SCD) where the attribute of a record changes over time and you want to maintain a history of these changes, you can use the `merge` write disposition with the scd2 strategy.


When using the merge disposition, you need to specify a `primary_key` or `merge_key` for the resource.


In [8]:
import dlt


@dlt.resource(
    name="pokemon",
    write_disposition="merge",
    primary_key="id",
)
def merge_pokemon() -> TDataItems:
    yield data


merge_pipeline = dlt.pipeline(
    pipeline_name="poke_pipeline_merge",
    destination="duckdb",
    dataset_name="pokemon_data",
)

load_info = merge_pipeline.run(merge_pokemon)
print(load_info)

# explore loaded data
merge_pipeline.dataset().pokemon.df()

Pipeline poke_pipeline_merge load step finished in 0.30 seconds
1 load package(s) were loaded to destination duckdb and into dataset pokemon_data
The duckdb destination used duckdb:////content/poke_pipeline_merge.duckdb location to store data
Load package 1795193288.863632 is LOADED and contains no failed jobs


,id,name,size__weight,size__height,_dlt_load_id,_dlt_id
0,25,pikachu,6.0,0.4,1795193288.863632,Y8O/RXlQmbCUWw
1,4,charmander,8.5,0.6,1795193288.863632,E5StP2RAKd58OA
2,1,bulbasaur,6.9,0.7,1795193288.863632,N1APK7WSzg3PzQ


The merge write disposition can be used with three different strategies:

* delete-insert (default strategy)
* scd2
* upsert

---
##  **Incremental Loading**


Incremental loading is the act of loading only new or changed data and not old records that we already loaded.

Imagine you’re a Pokémon trainer trying to catch ‘em all. You don’t want to keep visiting the same old PokéStops, catching the same old Bulbasaurs—you only want to find new and exciting Pokémon that have appeared since your last trip. That’s what incremental loading is all about: collecting only the new data that’s been added or changed, without wasting your Poké Balls (or database resources) on what you already have.

In this example, we have a dataset of Pokémon, each with a **unique ID**, their **name**, **size** (height and weight), and **when** they were "caught" (`created_at` field).

### **Step 1: Adding the `created_at` Field**

In [9]:
# We added `created_at` field to the data
created_data = [
    {
        "id": "1",
        "name": "bulbasaur",
        "size": {"weight": 6.9, "height": 0.7},
        "created_at": "2024-12-01",
    },
    {
        "id": "4",
        "name": "charmander",
        "size": {"weight": 8.5, "height": 0.6},
        "created_at": "2024-09-01",
    },
    {
        "id": "25",
        "name": "pikachu",
        "size": {"weight": 6, "height": 0.4},
        "created_at": "2023-06-01",
    },
]

**The goal**: Load only Pokémons caught after January 1, 2024, skipping the ones you already have.

### **Step 2: Defining the incremental logic**

Using `dlt`, we set up an [incremental filter](https://www.google.com/url?q=https://dlthub.com/docs/general-usage/incremental-loading%23incremental-loading-with-a-cursor-field&sa=D&source=editors&ust=1734717286675253&usg=AOvVaw3rAF3y3p86sGt49ImCTgon) to only fetch Pokémons caught after a certain date:
```python
cursor_date = dlt.sources.incremental("created_at", initial_value="2024-01-01")
```
This tells `dlt`:
- **Start date**: January 1, 2024 (`initial_value`).
- **Field to track**: `created_at` (our timestamp).

As you run the pipeline repeatedly, `dlt` will keep track of the latest `created_at` value processed. It will skip records older than this date in future runs.


In [10]:
import dlt


@dlt.resource(
    name="pokemon",
    write_disposition="append",
)
def incremental_pokemon(
    cursor_date: dlt.sources.incremental[str] = dlt.sources.incremental(
        "created_at", initial_value="2024-01-01"
    )
) -> TDataItems:
    yield created_data

We use the `@dlt.resource` decorator to declare table **name** to which data will be loaded and **write disposition**, which is **append** by default.

### **Step 3: Running the pipeline**
Finally, we run our pipeline and load the fresh Pokémon data:

In [11]:
incremental_pipeline = dlt.pipeline(
    pipeline_name="poke_pipeline_incremental",
    destination="duckdb",
    dataset_name="pokemon_data",
)

load_info = incremental_pipeline.run(incremental_pokemon)
print(load_info)

# explore loaded data
incremental_pipeline.dataset().pokemon.df()

Pipeline poke_pipeline_incremental load step finished in 0.20 seconds
1 load package(s) were loaded to destination duckdb and into dataset pokemon_data
The duckdb destination used duckdb:////content/poke_pipeline_incremental.duckdb location to store data
Load package 1795193367.1930096 is LOADED and contains no failed jobs


,id,name,created_at,size__weight,size__height,_dlt_load_id,_dlt_id
0,1,bulbasaur,2024-12-01,6.9,0.7,1795193367.1930096,y1jySaLLqX0qkw
1,4,charmander,2024-09-01,8.5,0.6,1795193367.1930096,Eoi2hcI8b8UYnw


This:
1. Loads **only Charmander and Bulbasaur** (caught after 2024-01-01).
2. Skips Pikachu because it’s old news.

Only data for 2024 year was loaded.

![Lesson_5_Write_disposition_and_incremental_loading_img2](https://storage.googleapis.com/dlt-blog-images/dlt-fundamentals-course/Lesson_5_Write_disposition_and_incremental_loading_img2.png)

Run the same pipeline again. The pipeline will detect that there are **no new records** based on the `created_at` field and the incremental cursor. As a result, **no new data will be loaded** into the destination:
>0 load package(s) were loaded

In [12]:
load_info = incremental_pipeline.run(incremental_pokemon)
print(load_info)

Pipeline poke_pipeline_incremental load step did not start
0 load package(s) were loaded to destination duckdb and into dataset None
The duckdb destination used duckdb:////content/poke_pipeline_incremental.duckdb location to store data


### **Why incremental loading matters**

* **Efficiency**. Skip redundant data, saving time and resources.
* **Scalability**. Handle growing datasets without bottlenecks.
* **Automation**. Let the tool track changes for you—no manual effort.

## **Update and deduplicate your data**
The script above finds new pokemons and adds them to the database. It will ignore any updates to user information.

In [13]:
# We added `updated_at` field to the data
updated_data = [
    {
        "id": "1",
        "name": "bulbasaur",
        "size": {"weight": 6.9, "height": 0.7},
        "created_at": "2024-12-01",
        "updated_at": "2024-12-01",
    },
    {
        "id": "4",
        "name": "charmander",
        "size": {"weight": 8.5, "height": 0.6},
        "created_at": "2024-09-01",
        "updated_at": "2024-09-01",
    },
    {
        "id": "25",
        "name": "pikachu",
        "size": {
            "weight": 9,
            "height": 0.4,
        },
        "created_at": "2023-06-01",
        "updated_at": "2024-12-16",
    },
]

Get always fresh content of all the pokemons: combine an **incremental load** with **merge** write disposition, like in the script below.

In [14]:
import dlt


@dlt.resource(
    name="pokemon",
    write_disposition="merge",
    primary_key="id",
)
def dedup_pokemon(
    data: TDataItems,
    cursor_date: dlt.sources.incremental[str] = dlt.sources.incremental(
        "updated_at", initial_value="2024-01-01"
    ),
) -> TDataItems:
    yield data

The incremental cursor keeps an eye on the `updated_at` field. Every time the pipeline runs, it only processes records with `updated_at` values greater than the last run.

In [15]:
dedup_pipeline = dlt.pipeline(
    pipeline_name="poke_pipeline_dedup",
    destination="duckdb",
    dataset_name="pokemon_data",
)

load_info = dedup_pipeline.run(dedup_pokemon(updated_data))
print(load_info)

# explore loaded data
dedup_pipeline.dataset().pokemon.df()

Pipeline poke_pipeline_dedup load step finished in 0.26 seconds
1 load package(s) were loaded to destination duckdb and into dataset pokemon_data
The duckdb destination used duckdb:////content/poke_pipeline_dedup.duckdb location to store data
Load package 1795193398.5635605 is LOADED and contains no failed jobs


,id,name,created_at,updated_at,size__weight,size__height,_dlt_load_id,_dlt_id
0,25,pikachu,2023-06-01,2024-12-16,9.0,0.4,1795193398.5635605,wUXbm+gHLptGkg
1,1,bulbasaur,2024-12-01,2024-12-01,6.9,0.7,1795193398.5635605,zqhp8fslpwHH8w
2,4,charmander,2024-09-01,2024-09-01,8.5,0.6,1795193398.5635605,RrA28hEnwP5s3w


All Pokémons are processed because this is the pipeline’s first run.

Now, let’s say Pikachu goes to gym and sheds some weight (down to 7.5), and the `updated_at` field is set to `2024-12-23`.

In [16]:
reupdated_data = [
    {
        "id": "1",
        "name": "bulbasaur",
        "size": {"weight": 6.9, "height": 0.7},
        "created_at": "2024-12-01",
        "updated_at": "2024-12-01",
    },
    {
        "id": "4",
        "name": "charmander",
        "size": {"weight": 8.5, "height": 0.6},
        "created_at": "2024-09-01",
        "updated_at": "2024-09-01",
    },
    {
        "id": "25",
        "name": "pikachu",
        "size": {"weight": 7.5, "height": 0.4},
        "created_at": "2023-06-01",
        "updated_at": "2024-12-23",
    },
]

Run the same pipeline:

In [17]:
load_info = dedup_pipeline.run(dedup_pokemon(reupdated_data))
print(load_info)

# explore loaded data
dedup_pipeline.dataset().pokemon.df()

Pipeline poke_pipeline_dedup load step finished in 0.20 seconds
1 load package(s) were loaded to destination duckdb and into dataset pokemon_data
The duckdb destination used duckdb:////content/poke_pipeline_dedup.duckdb location to store data
Load package 1795193414.276005 is LOADED and contains no failed jobs


,id,name,created_at,updated_at,size__weight,size__height,_dlt_load_id,_dlt_id
0,1,bulbasaur,2024-12-01,2024-12-01,6.9,0.7,1795193398.5635605,zqhp8fslpwHH8w
1,4,charmander,2024-09-01,2024-09-01,8.5,0.6,1795193398.5635605,RrA28hEnwP5s3w
2,25,pikachu,2023-06-01,2024-12-23,7.5,0.4,1795193414.276005,2Kj815KGcRPbKA


**What happened?**

* The pipeline detected that `updated_at` for Bulbasaur and Charmander hasn’t changed—they’re skipped.
* Pikachu’s record was updated to reflect the latest weight.

You can see that the **`_dlt_load_id`** for Bulbasaur and Charmander remained the same, but for Pikachu it was changed since only the updated Pikachu data was loaded into the destination.

The **`dlt.sources.incremental`** instance above has the following attributes:

* **`cursor_date.initial_value`** which is always equal to "2024-01-01" passed in the constructor;
* **`cursor_date.start_value`** a maximum `updated_at` value from the previous run or the `initial_value` on the first run;
* **`cursor_date.last_value`** a "real-time" `updated_at` value updated with each yielded item or page. Before the first yield, it equals `start_value`;
* **`cursor_date.end_value`** (not used here) marking the end of the backfill range.

## **Example**
You can use them in the resource code to make **more efficient requests**. Take look at the GitHub API example:

In [18]:
import os
from typing import Iterable
import dlt
from dlt.extract import DltResource
from dlt.sources.helpers import requests
from dlt.sources.helpers.rest_client import RESTClient
from dlt.sources.helpers.rest_client.auth import BearerTokenAuth
from dlt.sources.helpers.rest_client.paginators import HeaderLinkPaginator

from google.colab import userdata

os.environ["SOURCES__ACCESS_TOKEN"] = userdata.get("SECRET_KEY")


@dlt.source
def github_source(access_token: str = dlt.secrets.value) -> Iterable[DltResource]:
    client = RESTClient(
        base_url="https://api.github.com",
        auth=BearerTokenAuth(token=access_token),
        paginator=HeaderLinkPaginator(),
    )

    @dlt.resource(name="issues", write_disposition="merge", primary_key="id")
    def github_issues(
        cursor_date: dlt.sources.incremental[str] = dlt.sources.incremental(
            "updated_at", initial_value="2024-12-01"
        )
    ) -> TDataItems:
        params = {
            "since": (cursor_date.last_value),
            "status": "open",
        }
        for page in client.paginate("repos/dlt-hub/dlt/issues", params=params):
            yield page

    return github_issues


# define new dlt pipeline
pipeline = dlt.pipeline(pipeline_name="github_incr", destination="duckdb")


# run the pipeline with the new resource
load_info = pipeline.run(github_source())
print(load_info)

2026-11-20 16:52:25,320|[WARNING]|2773|134423922978944|dlt|validate.py|verify_normalized_table:113|In schema `github_source`: The following columns in table 'issues' did not receive any data during this load and therefore could not have their types inferred:
  - active_lock_reason
  - assignee
  - closed_at
  - closed_by
  - milestone
  - milestone__due_on
  - performed_via_github_app
  - pinned_comment
  - pull_request__merged_at
  - type

Unless type hints are provided, these columns will not be materialized in the destination.
One way to provide type hints is to use the 'columns' argument in the '@dlt.resource' decorator.  For example:

@dlt.resource(columns={'active_lock_reason': {'data_type': 'text'}})

2026-11-20 16:52:25,322|[WARNING]|2773|134423922978944|dlt|validate.py|verify_normalized_table:113|In schema `github_source`: The following columns in table 'issues__labels' did not receive any data during this load and therefore could not have their types inferred:
  - archived_at

Pipeline github_incr load step finished in 1.81 seconds
1 load package(s) were loaded to destination duckdb and into dataset github_incr_dataset
The duckdb destination used duckdb:////content/github_incr.duckdb location to store data
Load package 1795193536.5144215 is LOADED and contains no failed jobs


Pay attention to how we use the **since** GitHub API parameter and `cursor_date.last_value` to tell GitHub which issues we are interested in. `cursor_date.last_value` holds the last `cursor_date` value from the previous run.

Run the pipeline again and make sure that **no data is loaded**.

In [19]:
# run the pipeline with the new resource
load_info = pipeline.run(github_source())
print(load_info)

Pipeline github_incr load step did not start
0 load package(s) were loaded to destination duckdb and into dataset None
The duckdb destination used duckdb:////content/github_incr.duckdb location to store data


## **Apply Hints**

Alternatively, you can use `apply_hints` on a resource to define an incremental field:

```python
resource = resource()
resource.apply_hints(incremental=dlt.sources.incremental("updated_at"))
```

When you apply an incremental hint using `apply_hints`, the source still performs a full extract. The incremental hint is used by `dlt` to filter the data after it has been extracted, before it is loaded into the destination.

## **Exercise 1: Make the GitHub API pipeline incremental**

In the previous lessons, you built a pipeline to pull data from the GitHub API. Now, let’s level it up by making it incremental, so it fetches only new or updated data.


Transform your GitHub API pipeline to use incremental loading. This means:

* Implement a new `dlt.resource` for `pulls/comments` (List comments for Pull Requests) endpoint.
* Fetch only pulls comments updated after the last pipeline run.
* Use the `updated_at` field from the GitHub API as the incremental cursor.
* [Endpoint documentation](https://docs.github.com/en/rest/pulls/comments?apiVersion=2022-11-28#list-review-comments-in-a-repository)
* Endpoint URL: `https://api.github.com/repos/OWNER/REPO/pulls/comments`
* Use the `since` parameter - only show results that were last updated after the given time - and `last_value`.
* `initial_value` is `2024-12-01`.


### Question

How many columns does the `comments` table have?

In [20]:
import os
from typing import Iterable
import dlt
from dlt.extract import DltResource
from dlt.sources.helpers import requests
from dlt.sources.helpers.rest_client import RESTClient
from dlt.sources.helpers.rest_client.auth import BearerTokenAuth
from dlt.sources.helpers.rest_client.paginators import HeaderLinkPaginator

from google.colab import userdata

os.environ["SOURCES__ACCESS_TOKEN"] = userdata.get("SECRET_KEY")


@dlt.source
def github_source(access_token: str = dlt.secrets.value) -> Iterable[DltResource]:
    client = RESTClient(
        base_url="https://api.github.com",
        auth=BearerTokenAuth(token=access_token),
        paginator=HeaderLinkPaginator(),
    )

    @dlt.resource(name="comments", write_disposition="merge", primary_key="id")
    def github_comments(
        cursor_date: dlt.sources.incremental[str] = dlt.sources.incremental(
            "updated_at", initial_value="2024-12-01"
        )
    ) -> TDataItems:
        params = {
            "since": (cursor_date.last_value),
        }
        for page in client.paginate("repos/dlt-hub/dlt/pulls/comments", params=params):
            yield page

    return github_comments


# define new dlt pipeline
pipeline = dlt.pipeline(pipeline_name="github_incr", destination="duckdb")


# run the pipeline with the new resource
load_info = pipeline.run(github_source())
print(load_info)

Pipeline github_incr load step finished in 10.03 seconds
1 load package(s) were loaded to destination duckdb and into dataset github_incr_dataset
The duckdb destination used duckdb:////content/github_incr.duckdb location to store data
Load package 1795193671.2775457 is LOADED and contains no failed jobs


In [21]:
pipeline.dataset().comments.df()

,url,pull_request_review_id,id,node_id,diff_hunk,path,commit_id,original_commit_id,body,created_at,...,user__received_events_url,user__type,user__user_view_type,user__site_admin,_dlt_load_id,_dlt_id,original_start_line,start_side,start_line,copilot_code_review_severity
0,https://api.github.com/repos/dlt-hub/dlt/pulls...,2506053783,1886735511,PRRC_kwDOGvRYu85wdUyX,"@@ -52,6 +54,8 @@ class PipeIteratorConfigurat...",dlt/extract/pipe_iterator.py,9b9c9ece7cbb93b18f43b195f49bc2fa5ae4b243,9b9c9ece7cbb93b18f43b195f49bc2fa5ae4b243,could this be a callback? Or even better a gen...,2024-12-16 12:28:07+00:00,...,https://api.github.com/users/joscha/received_e...,User,public,False,1795193671.2775457,CinoHM6SFiqSTQ,<NA>,None,<NA>,None
1,https://api.github.com/repos/dlt-hub/dlt/pulls...,2506219264,1886834898,PRRC_kwDOGvRYu85wdtDS,"@@ -3960,3 +3961,58 @@ def some_data(\n \n ...",tests/extract/test_incremental.py,fc013f591698fba41a6c3eda2e1c32997f68385b,9d38f56d1f3a34cb6e721955e185b03cf4ad29d0,```suggestion\r\n p = dlt.pipeline(pipeline...,2024-12-16 13:40:24+00:00,...,https://api.github.com/users/joscha/received_e...,User,public,False,1795193671.2775457,qGqzjvSB792NFA,<NA>,None,<NA>,None
2,https://api.github.com/repos/dlt-hub/dlt/pulls...,2537473488,1907372889,PRRC_kwDOGvRYu85xsDNZ,"@@ -6,6 +6,8 @@ banned-modules = datetime = us...",tox.ini,84e78b91750a99f5e96d91003f9f96e105590e4c,8cebdf4063e4d745c75faf1a5dfd03bd1308b2d7,this is cool!,2025-01-08 15:27:57+00:00,...,https://api.github.com/users/rudolfix/received...,User,public,False,1795193671.2775457,0EuePKgPb0BkfA,<NA>,None,<NA>,None
3,https://api.github.com/repos/dlt-hub/dlt/pulls...,2562863866,1922722074,PRRC_kwDOGvRYu85ymmka,"@@ -102,6 +103,18 @@ def close_connection(self...",dlt/destinations/impl/databricks/sql_client.py,428c075644368682c61f98ccb6f7bb690d217f5b,2bd0be005c95bce1ec1844c752ef934469d137db,create volume ad hoc before uploading file,2025-01-20 17:48:18+00:00,...,https://api.github.com/users/rudolfix/received...,User,public,False,1795193671.2775457,LztNaQdEIZT/Ww,<NA>,None,<NA>,None
4,https://api.github.com/repos/dlt-hub/dlt/pulls...,2562863866,1922722518,PRRC_kwDOGvRYu85ymmrW,"@@ -1,4 +1,6 @@\n [runtime]\n+log_level=""DEBUG""",tests/.dlt/config.toml,428c075644368682c61f98ccb6f7bb690d217f5b,2bd0be005c95bce1ec1844c752ef934469d137db,remember to remove before final push,2025-01-20 17:48:58+00:00,...,https://api.github.com/users/rudolfix/received...,User,public,False,1795193671.2775457,3ql3h4MQqQ8flw,<NA>,None,<NA>,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4167,https://api.github.com/repos/dlt-hub/dlt/pulls...,5504901727,4259750050,PRRC_kwDOGvRYu8795qCi,"@@ -12,12 +12,12 @@ Planning to use `dlt` in p...",docs/website/docs/dlt-ecosystem/verified-sourc...,cec33ee992efeffd656c83d0161d17cbeb77a56e,10fc0448aca4716de991314737b2a5f8889eba87,>let's also link mssql Change Tracking\r\nWoul...,2026-10-13 14:14:13+00:00,...,https://api.github.com/users/zilto/received_ev...,User,public,False,1795193671.2775457,7MH1ZszDZhkv/Q,<NA>,None,<NA>,None
4168,https://api.github.com/repos/dlt-hub/dlt/pulls...,5531287103,4281542800,PRRC_kwDOGvRYu87_MyiQ,"@@ -0,0 +1,140 @@\n+---\n+title: REST API refe...",docs/website/docs/hub/reference/api/index.md,49e0b5c30098b5fe920a7f5c2f3e74410f2fff30,ef2759ff95f62527452611080fa1c96b4d5d5a7d,we can copy the icon that we also have for the...,2026-10-15 09:54:35+00:00,...,https://api.github.com/users/lis365b/received_...,User,public,False,1795193671.2775457,sJ+xxUeiDvSV5A,<NA>,None,<NA>,None
4169,https://api.github.com/repos/dlt-hub/dlt/pulls...,5651683712,4377025500,PRRC_kwDOGvRYu88AAAABBOQb3A,"@@ -0,0 +1,280 @@\n+""""""Generates the dltHub Ru...",docs/tools/generate_openapi_docs.py,49e0b5c30098b5fe920a7f5c2f3e74410f2fff30,ef2759ff95f62527452611080fa1c96b4d5d5a7d,Added in 2ce8c86a4: every operation now has a ...,2026-10-26 12:08:37+00:00,...,https://api.github.com/users/sh-rp/received_ev...,User,public,False,1795193671.2

✅ ▶ Proceed to the [next lesson](https://github.com/dlt-hub/dlt/blob/master/docs/education/dlt-fundamentals-course/lesson_6_how_dlt_works.ipynb)!